# Lab: RAG Evaluation Harness
## Learning Objectives
By the end of this lab, you will:
- Create a **Golden Dataset** (queries → ideal answer + retrieved context)
- Score **retrieval** with DeepEval's Contextual Precision / Recall / Relevancy metrics
- Score **generation** with DeepEval's Faithfulness and Answer Relevancy metrics
- Route every metric through a single **LiteLLM judge** (any provider)
- Read the judge's **reasons**, not just the scores, and gate on thresholds
- Record a **baseline** you will improve against in later sessions

> **Two halves, one ruler.** Part A measures **retrieval** (did the right context come back?).
> Part B measures **generation** (given that context, did the model use it faithfully?).
> Both halves now run on **DeepEval** — the same framework the capstone uses.
## Setup

In [ ]:
# DeepEval (LLM-as-judge) + LiteLLM. Use the kernel's pip (not uv) so it lands in this env.
!pip install -q -U deepeval litellm python-dotenv

In [ ]:
import os
import logging
from dotenv import load_dotenv
from deepeval.models import LiteLLMModel

# Keep output clean and opt out of DeepEval telemetry.
logging.getLogger("deepeval").setLevel(logging.ERROR)
os.environ["DEEPEVAL_TELEMETRY"] = "0"
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

# Loads OPENROUTER_API_KEY (and optionally EVAL_JUDGE_MODEL) from labs/.env
load_dotenv()

# One judge for every metric. Passing a LiteLLMModel object (not a bare string)
# keeps DeepEval on our provider instead of defaulting to OpenAI; any LiteLLM
# provider works via the model id.
MODEL_ID = os.getenv("EVAL_JUDGE_MODEL", "openrouter/deepseek/deepseek-v4-flash:free")
judge = LiteLLMModel(model=MODEL_ID, temperature=0)
print(f"Judge model: {MODEL_ID}")

## Part 1: The Golden Dataset

A **golden dataset** is a hand-curated set of ground-truth examples. For a DeepEval run, each example needs four fields:

- **`input`** — the user query
- **`expected_output`** — the ideal answer a domain expert would write (anchors Contextual Precision & Recall)
- **`actual_output`** — what *your* RAG system actually answered
- **`retrieval_context`** — the chunks your retriever returned for that query

Why it matters: **reproducibility** (compare experiments on fixed data), **regression detection** (did changing chunking/embeddings help or hurt?), and **human alignment** (experts define what "relevant" and "correct" mean).

Below, `CORPUS` maps a chunk id → its text so we can assemble `retrieval_context` from ids, just like a real retriever would return ranked chunks.

In [ ]:
# A tiny corpus: chunk id -> text. A real retriever returns ranked chunks like these.
CORPUS = {
    "doc_rag_intro_0": "Retrieval-Augmented Generation (RAG) combines a retriever with a generator: relevant documents are fetched from a knowledge base and passed to an LLM so its answer is grounded in external knowledge.",
    "doc_rag_overview_1": "By grounding responses in retrieved context, RAG reduces hallucination and lets a model answer questions about private or up-to-date data without retraining.",
    "doc_chunking_0": "Chunking splits documents into smaller passages before embedding so retrieval returns focused, relevant context instead of whole documents.",
    "doc_hybrid_0": "Hybrid search blends lexical BM25 scores with dense vector similarity, capturing both exact keyword matches and semantic meaning.",
    "doc_bm25_0": "BM25 is a bag-of-words ranking function that scores documents by term frequency and inverse document frequency.",
    "doc_eval_0": "Faithfulness measures whether every claim in an answer is supported by the retrieved context; answer relevancy measures whether the answer addresses the question.",
    "doc_noise_0": "The Eiffel Tower is located in Paris and was completed in 1889.",
    "doc_noise_1": "Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen.",
    "doc_noise_2": "The mitochondrion is the powerhouse of the cell.",
}

# Golden dataset: query -> ideal answer, the system's actual answer, and the relevant chunk ids.
golden_dataset = [
    {
        "query_id": "q1",
        "query": "What is Retrieval-Augmented Generation?",
        "expected_output": "RAG combines a retriever with an LLM: relevant documents are fetched and passed to the model so its answer is grounded in external knowledge, which reduces hallucination.",
        "answer": "Retrieval-Augmented Generation (RAG) fetches relevant documents and feeds them to an LLM so the answer is grounded in external knowledge, reducing hallucination [1].",
        "relevant_doc_ids": {"doc_rag_intro_0", "doc_rag_overview_1"},
    },
    {
        "query_id": "q2",
        "query": "How does hybrid search compare to BM25 alone?",
        "expected_output": "Hybrid search combines BM25 lexical scores with dense vector similarity, so it captures exact keyword matches and semantic meaning, whereas BM25 alone only matches terms.",
        "answer": "Hybrid search blends BM25 keyword scoring with dense vector similarity, capturing both exact matches and semantic meaning that BM25 alone misses [1][2].",
        "relevant_doc_ids": {"doc_hybrid_0", "doc_bm25_0"},
    },
    {
        "query_id": "q3",
        "query": "Why do we chunk documents before embedding?",
        "expected_output": "Chunking splits documents into smaller passages so retrieval returns focused, relevant context instead of entire documents.",
        "answer": "We chunk documents so the retriever can return small, focused passages instead of whole documents, which keeps the context relevant [1].",
        "relevant_doc_ids": {"doc_chunking_0"},
    },
]

print(f"Golden dataset: {len(golden_dataset)} queries, corpus of {len(CORPUS)} chunks")
for item in golden_dataset:
    print(f"  {item['query_id']}: {item['query']}")

## Part 2: Simulated Retrieval Results

To evaluate, we need what the retriever actually returned: an **ordered list of chunk ids** per query (best first), mixing relevant chunks with noise. This stands in for your real RAG service from Lab 3 — swap in its output when you run this for real.

We turn each ordered id list into a **`retrieval_context`** (the chunk *texts*, in rank order) — exactly the field DeepEval's metrics score.

In [ ]:
# Ordered chunk ids the retriever returned (best first) -- mixes relevant + noise.
simulated_retrievals = {
    "q1": ["doc_rag_intro_0", "doc_noise_0", "doc_rag_overview_1"],
    "q2": ["doc_noise_1", "doc_hybrid_0", "doc_bm25_0"],
    "q3": ["doc_chunking_0", "doc_noise_2", "doc_noise_0"],
}

def build_retrieval_context(query_id, k=3):
    """Map ordered chunk ids -> their texts (the retrieval_context DeepEval scores)."""
    ids = simulated_retrievals[query_id][:k]
    return [CORPUS[doc_id] for doc_id in ids]

for item in golden_dataset:
    qid = item["query_id"]
    ids = simulated_retrievals[qid]
    marks = ["*" if d in item["relevant_doc_ids"] else " " for d in ids]
    print(f"{qid}: " + " | ".join(f"{m}{d}" for m, d in zip(marks, ids)))
print("\n(* = relevant chunk)")

## Part 3: Scoring Retrieval with DeepEval

DeepEval scores retrieval with three **LLM-as-judge** metrics (no doc-id math — the judge reads the `expected_output` and the retrieved chunks):

| Metric | What it asks | Needs |
|--------|--------------|-------|
| **Contextual Precision** | Are the *relevant* chunks ranked above the noise? | expected_output + retrieval_context |
| **Contextual Recall** | Does the retrieved context cover everything in the ideal answer? | expected_output + retrieval_context |
| **Contextual Relevancy** | How much of the retrieved context is actually on-topic? | retrieval_context |

> Classic IR metrics (Hit Rate@K, MRR, Precision@K, Recall@K) are deterministic id-based checks and remain useful — DeepEval simply doesn't provide them. Here we standardize on the LLM-judged Contextual metrics so retrieval and generation share one framework and one judge.

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)

THRESHOLD = 0.7
contextual_precision = ContextualPrecisionMetric(threshold=THRESHOLD, model=judge, include_reason=True)
contextual_recall = ContextualRecallMetric(threshold=THRESHOLD, model=judge, include_reason=True)
contextual_relevancy = ContextualRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

# One LLMTestCase per query (retrieval_context = ranked chunk texts).
retrieval_test_cases = [
    LLMTestCase(
        input=item["query"],
        actual_output=item["answer"],
        expected_output=item["expected_output"],
        retrieval_context=build_retrieval_context(item["query_id"]),
    )
    for item in golden_dataset
]

# Runs the judge over every (test case x metric). This is your retrieval BASELINE.
retrieval_results = evaluate(
    test_cases=retrieval_test_cases,
    metrics=[contextual_precision, contextual_recall, contextual_relevancy],
)

## Part 4: Read the Reasons, Not Just the Scores

The biggest advantage of LLM-as-judge over id-based math is the **reason** attached to every score. A low Contextual Recall with the reason *"the ideal answer mentions semantic meaning but no retrieved chunk covers it"* tells you exactly what to fix. Let's pull those reasons out of the run above.

In [ ]:
# DeepEval returns one result per test case; each carries per-metric score + reason.
for tr in retrieval_results.test_results:
    print(f"\nQuery: {tr.input}")
    for m in tr.metrics_data:
        status = "PASS" if m.success else "FAIL"
        print(f"  [{status}] {m.name}: {m.score:.2f}")
        print(f"         reason: {m.reason}")

### Exercise A: Per-Query Summary

Aggregate scores hide which query is weak. Complete `per_query_analysis` to turn the DeepEval run into one summary row per query, then we flag the weakest.

In [ ]:
def per_query_analysis(eval_result):
    """Summarize each query's Contextual metric scores from a DeepEval run."""
    rows = []
    for tr in eval_result.test_results:
        # Each entry in tr.metrics_data has .name, .score, .success, .reason
        scores = {m.name: m.score for m in tr.metrics_data}
        succeeded = {m.name: m.success for m in tr.metrics_data}

        # TODO: pull the three contextual scores out of `scores`
        #   (keys: "Contextual Precision", "Contextual Recall", "Contextual Relevancy")
        contextual_precision = None
        contextual_recall = None
        contextual_relevancy = None

        # TODO: passed = True only if ALL three metrics succeeded (use succeeded.values())
        passed = None

        rows.append({
            "query": tr.input[:40],
            "contextual_precision": contextual_precision,
            "contextual_recall": contextual_recall,
            "contextual_relevancy": contextual_relevancy,
            "passed": passed,
        })
    return rows

analysis = per_query_analysis(retrieval_results)
from tests import checks
checks.check_lab_5_4(analysis)

print(f"{'Query':<42} {'C.Prec':>7} {'C.Rec':>7} {'C.Rel':>7} {'Pass':>6}")
print("-" * 72)
for r in analysis:
    status = "PASS" if r['passed'] else "FAIL"
    print(f"{r['query']:<42} {r['contextual_precision']:>7.2f} {r['contextual_recall']:>7.2f} {r['contextual_relevancy']:>7.2f} {status:>6}")

worst = min(analysis, key=lambda x: (x['contextual_recall'] if x['contextual_recall'] is not None else 0))
print(f"\nWeakest query (by Contextual Recall): {worst['query']}")

# Part B — Generation Evaluation

Retrieval told us the right context came back. Now we measure whether the LLM **used** that context faithfully to write the answer — same DeepEval framework, same judge, two new metrics.

## Part 5: Faithfulness & Answer Relevancy

Two LLM-as-judge metrics catch the two silent generation failures:

- **Faithfulness** — is every claim in the answer supported by the `retrieval_context`? (catches hallucination)
- **Answer Relevancy** — does the answer actually address the `input` question? (catches off-topic answers)

A perfectly faithful answer can still be irrelevant, and vice-versa — that's why we score both. Below we run them on a clearly faithful answer and a hallucinated one, and read the judge's reasoning.

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

faithfulness = FaithfulnessMetric(threshold=THRESHOLD, model=judge, include_reason=True)
answer_relevancy = AnswerRelevancyMetric(threshold=THRESHOLD, model=judge, include_reason=True)

demo_cases = [
    LLMTestCase(
        input="What is RAG?",
        actual_output="RAG combines retrieval with generation to ground responses in external knowledge, reducing hallucination [1].",
        retrieval_context=["RAG combines retrieval with generation to ground LLM responses in external knowledge, reducing hallucinations."],
    ),
    LLMTestCase(
        input="What is RAG?",
        actual_output="RAG was invented by Facebook in 2020 and always uses FAISS for vector search.",
        retrieval_context=["RAG combines retrieval with generation."],
    ),
]

for tc in demo_cases:
    faithfulness.measure(tc)
    answer_relevancy.measure(tc)
    print(f"\nAnswer: {tc.actual_output[:70]}...")
    print(f"  Faithfulness:     {faithfulness.score:.2f} — {faithfulness.reason}")
    print(f"  Answer Relevancy: {answer_relevancy.score:.2f} — {answer_relevancy.reason}")

## Part 6: Gating CI with `assert_test`

The cells above use `metric.measure()` to inspect scores interactively. In CI you instead use `assert_test`, which **fails the build** when a metric falls below threshold:

```python
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import LiteLLMModel

# Same judge the course uses; a bare model string would fall back to OpenAI.
judge = LiteLLMModel(model="openrouter/deepseek/deepseek-v4-flash:free", temperature=0)
faithfulness = FaithfulnessMetric(threshold=0.7, model=judge)
answer_relevancy = AnswerRelevancyMetric(threshold=0.7, model=judge)

def test_rag_answer():
    test_case = LLMTestCase(
        input="What is RAG?",
        actual_output="RAG combines retrieval with generation [1].",
        retrieval_context=["RAG combines retrieval with generation..."],
    )
    assert_test(test_case, [faithfulness, answer_relevancy])
```

> **Cost:** an LLM judge runs a model per metric per case — roughly \$0.01–0.05 each on paid models, free on the DeepSeek model above. Keep golden sets small and gate only on the metrics that matter.

### Exercise B: Build a Generation Eval Suite

Complete `run_eval_suite` to score a batch of RAG responses with Faithfulness and Answer Relevancy and decide pass/fail against the threshold.

In [ ]:
def run_eval_suite(rag_responses, threshold=0.7):
    """Score a batch of RAG responses with DeepEval's generation metrics."""
    results = []
    for resp in rag_responses:
        test_case = LLMTestCase(
            input=resp["query"],
            actual_output=resp["answer"],
            retrieval_context=resp["retrieval_context"],
        )

        # TODO: measure faithfulness on test_case, then read faithfulness.score
        faith = None

        # TODO: measure answer relevancy on test_case, then read answer_relevancy.score
        relevancy = None

        # TODO: passed = True only if BOTH scores >= threshold
        passed = None

        results.append({
            "query": resp["query"][:40],
            "faithfulness": faith,
            "relevancy": relevancy,
            "passed": passed,
        })
    return results

test_responses = [
    {
        "query": "Why do we chunk documents?",
        "answer": "We split documents into smaller passages so retrieval returns focused, relevant context.",
        "retrieval_context": [CORPUS["doc_chunking_0"]],
    },
    {
        "query": "What does faithfulness measure?",
        "answer": "Faithfulness checks that every claim in the answer is supported by the retrieved context.",
        "retrieval_context": [CORPUS["doc_eval_0"]],
    },
]

suite_results = run_eval_suite(test_responses)
from tests import checks
checks.check_lab_5_6(suite_results)

print(f"{'Query':<42} {'Faith':>7} {'Relev':>7} {'Pass':>6}")
print("-" * 65)
for r in suite_results:
    status = "PASS" if r['passed'] else "FAIL"
    print(f"{r['query']:<42} {r['faithfulness']:>7.2f} {r['relevancy']:>7.2f} {status:>6}")

pass_rate = sum(1 for r in suite_results if r['passed']) / len(suite_results)
print(f"\nPass rate: {pass_rate:.0%}")

## Reflection Questions

**Retrieval**
1. **Metric Choice**: A legal discovery system must surface ALL relevant precedents. Which DeepEval metric matters most — Contextual Precision or Contextual Recall? Why?
2. **Golden Dataset Size**: How many examples do you need for a trustworthy baseline? What are the trade-offs of too few vs too many, given each example costs a judge call *per metric*?
3. **Baseline Discipline**: You recorded a retrieval baseline above. Next session adds hybrid search and re-ranking. Which Contextual metric do you expect each technique to move most, and why?

**Generation**
4. **LLM-as-Judge**: What are the risks of using a strong LLM to judge another LLM's outputs (position bias, verbosity preference, self-preference)? How would you mitigate them?
5. **Faithfulness vs Relevancy**: An answer can be perfectly faithful to the context yet useless to the user. Give an example, and say which metric catches it.
6. **Regression Gate**: You wire this suite into CI with `assert_test`. What faithfulness threshold would you set, and what do you do when a legitimate change trips it?

*Your answers here:*
1. ...
2. ...
3. ...
4. ...
5. ...
6. ...